In [1]:
%load_ext autoreload
%autoreload 2

# 6. LightGBM Baseline Training

Train baseline LightGBM model.

In [2]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np


sys.path.insert(0, str(Path.cwd().parent))
from src.data.loader import load_features
from src.models.splitting import split_train_holdout, split_features_target
from src.models.lightgbm_model import LightGBMModel
from src.utils.helpers import save_pickle, select_prediction_samples


## Load

In [3]:
# Configure experiment
feature_path = Path('../data/features')
artifact_dir = Path('../experiments/exp_001_lgbm/artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)

store_filter = ['CA_1', 'TX_1']
split_test_date = pd.Timestamp('2016-04-24') #-28 days from the last date in the dataset
split_validation_date = pd.Timestamp('2016-03-27') #-28 days from the test date
target_col = 'sales'
drop_cols = ['id', 'date', target_col]

print(f'Loading features from: {feature_path}')
df = load_features(feature_path, store_filter=store_filter)

# Holdout data for testing and training/validation 
train_val_df, test_df = split_train_holdout(df, split_date=split_test_date)
train_df, val_df = split_train_holdout(train_val_df, split_date=split_validation_date)

# Prepare data for training and validation
X_train, y_train = split_features_target(train_df, target_col=target_col, drop_cols=drop_cols)
X_val, y_val = split_features_target(val_df, target_col=target_col, drop_cols=drop_cols)

print('Split summary:')
print(f'  Train rows:   {len(train_df):,}')
print(f'  Val rows:     {len(val_df):,}')
print(f'  Test rows:    {len(test_df):,}')
print(f'  Features:     {X_train.shape[1]:,}')



Loading features from: ..\data\features
Applied store filter: ['CA_1', 'TX_1']
Split summary:
  Train rows:   11,494,730
  Val rows:     170,744
  Test rows:    170,744
  Features:     22


## Train and Test

In [4]:
# Train LightGBM model
model = LightGBMModel(name='lightgbm_exp001')
model.train(
    X_train,
    y_train,
    X_val=X_val,
    y_val=y_val,
    num_rounds=500,
    early_stopping=50,
 )

feature_importance = model.get_feature_importance(
    importance_type='gain',
    top_k=20
)

Training lightgbm_exp001 with 11494730 samples and 22 features...
Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 2.01608	valid's rmse: 1.8326
[200]	train's rmse: 1.9825	valid's rmse: 1.82642
Early stopping, best iteration is:
[203]	train's rmse: 1.98184	valid's rmse: 1.82638
[+] Model training complete! (num_boost_round: 203)


In [5]:
# Prepare data for testing
X_test, y_test = split_features_target(test_df, target_col=target_col, drop_cols=drop_cols)

# Remove unavailable sales information from test features
test_dates = test_df['date']

# Rolling means depend on recent sales from the test period
for col in ['rolling_mean_7', 'rolling_mean_28']:
    X_test.loc[test_dates > test_dates.min(), col] = np.nan

# Lag 7 becomes unavailable after the first 7 forecast days
X_test.loc[test_dates >= test_dates.min() + pd.Timedelta(days=7),'sales_lag_7'] = np.nan

# Recursive forecast
predictions = []
x_test_predict_parts = []
sales_history = train_val_df[['id', 'date', 'sales']].copy()

prediction_parts = []

for current_date in sorted(test_df['date'].unique()):

    print(f'  Predicting:   {current_date}')
    
    current_mask = test_df['date'] == current_date
    current_test = test_df.loc[current_mask].copy()
    X_current = X_test.loc[current_mask].copy()

    # Available sales history for each item
    history = sales_history.set_index(['id', 'date'])['sales']

    # Recalculate lag 7
    lag_date = current_date - pd.Timedelta(days=7)

    X_current['sales_lag_7'] = [
        history.get((item_id, lag_date), np.nan)
        for item_id in current_test['id']
    ]

    # Recalculate recent rolling means
    recent_history = (
        sales_history[sales_history['date'] < current_date]
        .sort_values(['id', 'date'])
    )

    rolling_7 = (
        recent_history.groupby('id')['sales']
        .apply(lambda x: x.tail(7).mean())
    )

    rolling_28 = (
        recent_history.groupby('id')['sales']
        .apply(lambda x: x.tail(28).mean())
    )

    X_current['rolling_mean_7'] = current_test['id'].map(rolling_7).values
    X_current['rolling_mean_28'] = current_test['id'].map(rolling_28).values

    # Store exactly the same matrix used by model.predict
    x_test_predict_parts.append(X_current.copy())

    # Predict current day
    y_hat = model.predict(X_current)

    current_predictions = current_test[
        ['id', 'date', target_col]
    ].copy()

    current_predictions = current_predictions.rename(
        columns={target_col: 'y_true'}
    )

    current_predictions['y_pred'] = y_hat

    prediction_parts.append(current_predictions)

    predictions.extend(y_hat)

    # Add predictions to history for next forecast days
    predicted_sales = current_test[['id', 'date']].copy()
    predicted_sales['sales'] = y_hat

    sales_history = pd.concat(
        [sales_history, predicted_sales],
        ignore_index=True
    )

X_test_predict = pd.concat(x_test_predict_parts, ignore_index=True)
X_test_predict = X_test_predict[X_train.columns]

predictions_df = pd.concat(
    prediction_parts,
    ignore_index=True
)

print(f'Predictions:     {len(predictions_df):,} rows')
print(predictions_df.head(10))

  Predicting:   2016-04-25 00:00:00
  Predicting:   2016-04-26 00:00:00
  Predicting:   2016-04-27 00:00:00
  Predicting:   2016-04-28 00:00:00
  Predicting:   2016-04-29 00:00:00
  Predicting:   2016-04-30 00:00:00
  Predicting:   2016-05-01 00:00:00
  Predicting:   2016-05-02 00:00:00
  Predicting:   2016-05-03 00:00:00
  Predicting:   2016-05-04 00:00:00
  Predicting:   2016-05-05 00:00:00
  Predicting:   2016-05-06 00:00:00
  Predicting:   2016-05-07 00:00:00
  Predicting:   2016-05-08 00:00:00
  Predicting:   2016-05-09 00:00:00
  Predicting:   2016-05-10 00:00:00
  Predicting:   2016-05-11 00:00:00
  Predicting:   2016-05-12 00:00:00
  Predicting:   2016-05-13 00:00:00
  Predicting:   2016-05-14 00:00:00
  Predicting:   2016-05-15 00:00:00
  Predicting:   2016-05-16 00:00:00
  Predicting:   2016-05-17 00:00:00
  Predicting:   2016-05-18 00:00:00
  Predicting:   2016-05-19 00:00:00
  Predicting:   2016-05-20 00:00:00
  Predicting:   2016-05-21 00:00:00
  Predicting:   2016-05-22 0

## Save Results

In [6]:
# Save model and results to experiment artifacts
artifact_dir.mkdir(parents=True, exist_ok=True)

if isinstance(store_filter, (list, tuple, set, pd.Index)):
    store_suffix = '_'.join(map(str, store_filter))
else:
    store_suffix = store_filter if store_filter else 'all_stores'

# Save X_train and X_test_predict for future reference
x_train_path = artifact_dir / f'X_train_{store_suffix}.parquet'
X_test_predict_path = artifact_dir / f'X_test_predict_{store_suffix}.parquet'
X_train.to_parquet(x_train_path, index=False)
X_test_predict.to_parquet(X_test_predict_path, index=False)

print(f'✓ X_train and X_test_predict saved to {x_train_path} and {X_test_predict_path}')

# Save model
model_path = artifact_dir / f'lightgbm_{store_suffix}.pkl'
save_pickle(model.get_model(), model_path)

# Save feature importance
importance_path = artifact_dir / f'feature_importance_{store_suffix}.parquet'
feature_importance.to_parquet(importance_path, index=False)

print(f'✓ Feature importance saved to {importance_path}')

# Save predictions
results_path = artifact_dir / f'predictions_{store_suffix}.parquet'
predictions_df.to_parquet(results_path, index=False)


print(f'✓ Predictions saved to {results_path}')

✓ X_train and X_test_predict saved to ..\experiments\exp_001_lgbm\artifacts\X_train_CA_1_TX_1.parquet and ..\experiments\exp_001_lgbm\artifacts\X_test_predict_CA_1_TX_1.parquet
Saved to ..\experiments\exp_001_lgbm\artifacts\lightgbm_CA_1_TX_1.pkl
✓ Feature importance saved to ..\experiments\exp_001_lgbm\artifacts\feature_importance_CA_1_TX_1.parquet
✓ Predictions saved to ..\experiments\exp_001_lgbm\artifacts\predictions_CA_1_TX_1.parquet


## Select and save sample

In [7]:
selected_predictions = select_prediction_samples(
    predictions=predictions_df,
    n_good=100,
    n_bad=100,
    max_per_item=3,
    zero_prediction_threshold=0.5,
    random_state=42
)

# Save sample
sample_path = artifact_dir / f'sample_{store_suffix}.parquet'
selected_predictions.to_parquet(sample_path, index=False)

print(f'✓ Sample saved to {sample_path}')

✓ Sample saved to ..\experiments\exp_001_lgbm\artifacts\sample_CA_1_TX_1.parquet
